# Part A — BEFORE: Claude Code + Snowflake-managed MCP
## Notebook 02

In the **BEFORE** state, an external client (Claude Code / Claude Desktop / claude.ai) is the *brain*.
It reaches into Snowflake through the **Snowflake-managed MCP server**, which exposes the Part 0 tools
(Cortex Analyst semantic view, Cortex Search, governed UDF) over a standards-based interface with OAuth.

This BEFORE state **works** — that matters. The point of this notebook is not that it is slow; it is that
when the reasoning loop lives **outside** the data plane, you lose control over three of the four pillars:

- **Governance** — the OAuth session runs as one `DEFAULT_ROLE`; there is no per-request RBAC on the
  reasoning, and the agent's *behavior* (what it will and won't do) is defined in the external client.
- **Cost** — there is no budget object you can bind to the external brain; its tokens are billed by
  Anthropic, outside Snowflake, so you cannot cap, tag, or chargeback the reasoning loop natively.
- **Observability** — Snowflake sees only the tool call it is handed; the planning/reasoning that decided
  to make that call happens on Anthropic's side and emits **no** server-side span here.

This notebook builds everything on the **Snowflake side** (MCP server object, OAuth security integration,
session defaults) and then makes the control gap concrete. The **Claude side** (adding the custom connector
and approving OAuth) can't be automated — it uses your personal Claude account — so it's an exact
step-list at the end.

> The honest BEFORE story is a **control** story, not a speed story. Claude's latency and token cost live
> on Anthropic's side, outside Snowflake, so the comparison focuses on the governance, budget, and trace
> controls you gain when the reasoning loop moves in-plane — and treats in-plane latency as an observable
> metric, not a head-to-head speed number.

**Prerequisite:** `gtm-01-foundation` completed.

---
## Section 1: Connect

In [ ]:
from snowflake.snowpark.context import get_active_session
import json

session = get_active_session()
session.sql("USE ROLE SYSADMIN").collect()
session.sql("USE DATABASE GTMAGENTS").collect()
session.sql("USE SCHEMA GTMAGENTS.DEMO").collect()
session.sql("USE WAREHOUSE GTMAGENTS_WH").collect()
print("Connected to GTMAGENTS.DEMO")

---
## Section 2: Create the Snowflake-managed MCP server

`CREATE MCP SERVER ... FROM SPECIFICATION` wraps existing Snowflake objects as MCP tools. We expose three:
the Cortex Analyst semantic view (`CORTEX_ANALYST_MESSAGE`), the Cortex Search service
(`CORTEX_SEARCH_SERVICE_QUERY`), and the governed UDF (`GENERIC`). MCP clients discover and invoke these
after authenticating. **Access to the server ≠ access to the tools** — each underlying object is granted
separately (least privilege).

In [ ]:
session.sql("""
CREATE OR REPLACE MCP SERVER GTMAGENTS_MCP
  FROM SPECIFICATION $$
tools:
  - name: \"gtm-analyst\"
    type: \"CORTEX_ANALYST_MESSAGE\"
    identifier: \"GTMAGENTS.DEMO.EMAIL_GTM_SV\"
    description: \"Answer NL questions about GTM sales-email performance by rep, team, region, quality tier, month.\"
    title: \"GTM Email Analyst\"
  - name: \"framework-search\"
    type: \"CORTEX_SEARCH_SERVICE_QUERY\"
    identifier: \"GTMAGENTS.DEMO.FRAMEWORK_SEARCH\"
    description: \"Retrieve best-practice email framework guidance.\"
    title: \"Email Framework Search\"
  - title: \"Team Performance Tool\"
    identifier: \"GTMAGENTS.DEMO.GTM_TEAM_PERFORMANCE\"
    name: \"gtm_team_performance\"
    type: \"GENERIC\"
    description: \"Return curated aggregate GTM performance metrics for a region (empty = all).\"
    config:
      type: \"function\"
      warehouse: \"GTMAGENTS_WH\"
      input_schema:
        type: \"object\"
        properties:
          REGION_FILTER:
            description: \"AMER, EMEA, APAC, or empty for all.\"
            type: \"string\"
  $$
""").collect()
print("MCP server GTMAGENTS_MCP created")

In [ ]:
# The server lists exactly the three tools we exposed
session.sql("DESCRIBE MCP SERVER GTMAGENTS_MCP").show(max_width=200)

---
## Section 3: OAuth security integration  (⚠️ ACCOUNTADMIN)

The MCP server authenticates external clients with Snowflake's built-in OAuth 2.0 service. One security
integration (client id + secret) is shared across all users in the account; each user still signs in
individually. **No dynamic client registration** — the redirect URI must match what the client shows.

This cell needs `ACCOUNTADMIN` and mutates account state, so it is **opt-in**: set `CREATE_OAUTH = True`
to run it. `OAUTH_CLIENT = CUSTOM`, name is UPPERCASE, redirect URI must be TLS.

In [ ]:
CREATE_OAUTH = False  # set True to (re)create the OAuth integration (ACCOUNTADMIN)

if CREATE_OAUTH:
    session.sql("USE ROLE ACCOUNTADMIN").collect()
    session.sql("""
        CREATE OR REPLACE SECURITY INTEGRATION GTMAGENTS_MCP_OAUTH
          TYPE = OAUTH
          OAUTH_CLIENT = CUSTOM
          ENABLED = TRUE
          OAUTH_CLIENT_TYPE = 'CONFIDENTIAL'
          OAUTH_REDIRECT_URI = 'https://claude.ai/api/mcp/auth_callback'
          COMMENT = 'OAuth client for the GTMAGENTS_MCP Snowflake-managed MCP server.'
    """).collect()
    session.sql("USE ROLE SYSADMIN").collect()
    print("OAuth integration created.")
else:
    print("Skipped (CREATE_OAUTH=False). Integration was created in setup validation.")

# Note for Claude Desktop: its callback is a localhost (non-TLS) URI. To support it, add
#   OAUTH_ALTERNATE_REDIRECT_URIS = ('http://localhost:PORT/...') and set
#   OAUTH_ALLOW_NON_TLS_REDIRECT_URI = TRUE on the integration.

In [ ]:
# Retrieve the client id + secret to paste into the Claude custom connector.
# (ACCOUNTADMIN required. Name is case-sensitive UPPERCASE.)
session.sql("USE ROLE ACCOUNTADMIN").collect()
secrets = session.sql("SELECT SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('GTMAGENTS_MCP_OAUTH') AS s").collect()[0]['S']
session.sql("USE ROLE SYSADMIN").collect()
s = json.loads(secrets)
print('OAUTH_CLIENT_ID    :', s['OAUTH_CLIENT_ID'])
print('OAUTH_CLIENT_SECRET:', s['OAUTH_CLIENT_SECRET'][:6] + '... (copy full value from output)')
print('(store the secret securely — treat like a password)')

---
## Section 4: MCP endpoint URL + session defaults

The MCP endpoint URL follows a fixed shape. **Underscore gotcha:** some MCP clients mis-parse hostnames with
underscores, so we use the hyphenated org-account URL form and an underscore-free database name (`GTMAGENTS`).

The MCP OAuth session uses the connecting user's **`DEFAULT_ROLE`** (secondary roles unsupported) and fails
to initialize if **`DEFAULT_WAREHOUSE`** is null — the classic *"list tools fails / no warehouse"* error.

In [ ]:
acct_url = session.sql("SELECT CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME() AS a").collect()[0]['A']
acct_url = acct_url.lower().replace('_', '-')  # hyphenate to dodge the underscore gotcha
mcp_url = f"https://{acct_url}.snowflakecomputing.com/api/v2/databases/GTMAGENTS/schemas/DEMO/mcp-servers/GTMAGENTS_MCP"
print('MCP endpoint URL (paste into Claude):')
print(' ', mcp_url)

In [ ]:
# Set DEFAULT_ROLE + DEFAULT_WAREHOUSE for the MCP user. Opt-in: mutates the user object.
APPLY_DEFAULTS = False  # set True to apply
MCP_USER = session.sql("SELECT CURRENT_USER() AS u").collect()[0]['U']

if APPLY_DEFAULTS:
    session.sql("USE ROLE ACCOUNTADMIN").collect()
    session.sql(f"ALTER USER \"{MCP_USER}\" SET DEFAULT_ROLE = 'GTMAGENTS_ROLE' DEFAULT_WAREHOUSE = 'GTMAGENTS_WH'").collect()
    session.sql("USE ROLE SYSADMIN").collect()
    print(f"Set DEFAULT_ROLE/DEFAULT_WAREHOUSE on {MCP_USER}")
else:
    print(f"Skipped (APPLY_DEFAULTS=False). Ensure {MCP_USER} has DEFAULT_ROLE + DEFAULT_WAREHOUSE before connecting Claude.")

---
## Section 5: The control gap — what Snowflake can and cannot see (BEFORE)

When the brain is external, Snowflake sees a tool call arrive, executes it, and returns the result. It does
**not** see the reasoning that produced the call. So the only thing observable in-plane is the tool's own
execution — the *reasoning loop*, the budget, and the routing decisions all live outside the perimeter.

The cell below makes this concrete: for a representative tool call we show exactly the slice Snowflake can
account for (the object + its grant), and name the three things it structurally **cannot** give you in this
architecture. This is the mirror image of the AFTER state (`gtm-03`), where every one of these becomes a
first-class, inspectable, in-plane object.

In [ ]:
# The control gap, made concrete. No timing here on purpose: the external brain's latency and token cost
# are billed by Anthropic, outside Snowflake, so there is no honest in-plane number to report.

# VISIBLE in-plane: the governed tool object and exactly who may use it. This is all Snowflake accounts for
# when an external brain calls a tool.
print("VISIBLE to Snowflake for an external tool call — the object and its grant:")
session.sql(
    "SHOW GRANTS ON VIEW GTMAGENTS.DEMO.EMAIL_GTM_SV"
).select('"privilege"', '"granted_to"', '"grantee_name"').show(max_width=200)

# INVISIBLE / UNGOVERNED in-plane when the brain is external:
print("""
NOT available in-plane in the BEFORE architecture:
  1. Reasoning trace   — the planning that chose this tool runs on Anthropic; it emits no server-side span.
  2. Budget control    — no orchestration budget object binds to the external brain; its tokens are billed
                         by Anthropic, so you cannot cap / tag / chargeback the reasoning loop natively.
  3. Per-request RBAC  — the OAuth session runs as one DEFAULT_ROLE; behavior is governed in the client.

In gtm-03 each of these becomes a first-class, inspectable Snowflake object (span, budget, grant).
""")

---
## Section 6: Connect Claude to the MCP server  (manual — your Claude account)

These steps **cannot be automated** — they run in your personal Claude account against the OAuth integration
and MCP URL built above.

1. In Claude, open **Settings → Connectors**.
2. Click **Add custom connector**.
3. **Name:** `Snowflake GTM`.  **MCP Server URL:** paste the URL printed in Section 4.
4. Paste the **OAUTH_CLIENT_ID** and **OAUTH_CLIENT_SECRET** from Section 3.
5. Click **Add**. Claude opens a browser window → sign in to Snowflake → approve the OAuth consent screen.
6. The three Snowflake tools (`gtm-analyst`, `framework-search`, `gtm_team_performance`) now appear in Claude.

### Demo prompts to run in Claude (BEFORE state)
- *"Using the GTM Email Analyst, what's the reply rate and win rate by region?"*
- *"Search the email framework for guidance on writing a clear call to action."*
- *"Call the team performance tool for the AMER region and summarize the top team."*

### Talking-point gotchas — the connector-reliability tax (Governance & Reliability pillar)
Every item below is operational surface you own **only** because the brain is external. In the AFTER
state (CoWork) there is no connector to configure, so this entire list disappears:
- **Redirect URI must match** what Claude displays, or OAuth fails.
- **DEFAULT_ROLE / DEFAULT_WAREHOUSE** must be set, or tool discovery fails with a warehouse error.
- **No underscores** in the account host / DB name in the URL (we use the hyphenated org-account form).
- **Network policy:** if enabled, allow-list the client provider's outbound IPs (e.g. Anthropic's) with an
  INGRESS network rule, or the connection is blocked.
- **PrivateLink** limits external client reachability; **PAT** is a fallback auth method (least-privilege role).
- **Data locality:** the external brain sees only what the tools return — but those results **leave the
  Snowflake perimeter** on every call, and the reasoning that requested them is neither budgeted nor traced.

---
## Summary

The BEFORE state is wired: a Snowflake-managed MCP server exposes the Part 0 tools, OAuth is configured, and
a latency/cost baseline is logged. Claude drives these tools as an **external brain** — powerful, but every
call pays a network + planning round-trip, and governance/observability live outside Snowflake. Next,
**`gtm-03-after-agents`** moves the brain **into the data plane** with a multi-agent Cortex Agents architecture.